# Charged-particle track reconstruction as a QAOA problem

This tutorial reconstructs two tracks in a toy three-layer detector. Candidate triplets of hits become binary variables $T_i$. Smooth candidates receive a reward, while pairs that reuse a hit receive a penalty:

$$
E(T)=-\sum_i s_iT_i+\lambda\sum_{i<j}C_{ij}T_iT_j.
$$

$C_{ij}=1$ when candidates $i$ and $j$ conflict. The resulting QUBO is solved with QARP's QAOA and checked against exhaustive enumeration.

**Encoding and scope.** There is one qubit per candidate triplet, not one amplitude per possible event. Candidate generation and scoring remain classical. In realistic detectors the candidate count can itself grow combinatorially, so this toy is an optimization-mapping tutorial rather than a scalability or advantage claim. The related LUXE study reported that QAOA did not outperform its classical baselines on the investigated instances; that negative result is useful context.

References: the [HEP quantum-computing review](https://arxiv.org/abs/2307.03236) and [Quantum Algorithms for Charged Particle Track Reconstruction in the LUXE Experiment](https://arxiv.org/abs/2304.01690).

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np

import qarp
from qarp.algorithms import QAOA, Sampler
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator
from qarp.optimizers import ScipyOptimizer
from qarp.resources import Stage, estimate

## 1. Detector hits and candidate triplets

The detector has three circular layers. Tracks A and B bend gently in azimuth; N labels unrelated noise hits. We assume a classical geometric preselection has rejected the N hits, then form **all** $2^3=8$ one-hit-per-layer triplets from the retained A/B hits. Two triplets are genuine and six are mixed-hit fakes. This makes the candidate-generation rule explicit while keeping the circuit workshop-sized.

In [ ]:
radii = np.array([1.0, 2.0, 3.0])
hits = {
    "A0": (0, 0.20),
    "A1": (1, 0.25),
    "A2": (2, 0.30),
    "B0": (0, -0.55),
    "B1": (1, -0.50),
    "B2": (2, -0.45),
    "N0": (0, 1.20),
    "N1": (1, -1.00),
    "N2": (2, 0.95),
}
retained_hits_by_layer = [
    ("A0", "B0"),
    ("A1", "B1"),
    ("A2", "B2"),
]
candidates = list(itertools.product(*retained_hits_by_layer))
assert len(candidates) == 8

fig, ax = plt.subplots(figsize=(6, 6))
for radius in radii:
    ax.add_patch(plt.Circle((0, 0), radius, fill=False, color="0.75"))
for name, (layer, phi) in hits.items():
    radius = radii[layer]
    x, y = radius * np.cos(phi), radius * np.sin(phi)
    color = "tab:gray" if name.startswith("N") else "tab:blue"
    ax.scatter(x, y, color=color)
    ax.text(1.04 * x, 1.04 * y, name)
ax.set(xlabel="x", ylabel="y", title="Three-layer toy detector")
ax.set_aspect("equal")
ax.set_xlim(-3.4, 3.4)
ax.set_ylim(-3.4, 3.4)
plt.show()

## 2. Score candidates and construct the conflict graph

For each triplet we fit $\phi(r)=a+br$ and use the RMS residual as a simple curvature-consistency proxy. The score $s_i=\exp[-(\mathrm{RMS}_i/0.12)^2]$ is near one for the two smooth tracks. A conflict is purely combinatorial: two candidates share at least one hit.

In [ ]:
def angular_residual(candidate):
    phis = np.array([hits[name][1] for name in candidate])
    fitted_phis = np.polyval(np.polyfit(radii, phis, 1), radii)
    return np.sqrt(np.mean((phis - fitted_phis) ** 2))


residuals = np.array([angular_residual(candidate) for candidate in candidates])
scores = np.exp(-((residuals / 0.12) ** 2))
conflicts = {
    (i, j)
    for i in range(len(candidates))
    for j in range(i + 1, len(candidates))
    if set(candidates[i]) & set(candidates[j])
}

for i, (candidate, residual, score) in enumerate(zip(candidates, residuals, scores, strict=True)):
    print(f"T{i}: {candidate}, residual={residual:.4f}, score={score:.4f}")
# print("Conflicting pairs:", sorted(conflicts))

The penalty $\lambda=0.6$ exceeds the largest fake-track reward. It suppresses reusing hits without overwhelming the two compatible true-track rewards.

In [ ]:
def qubo_to_ising(linear, quadratic):
    """Map E(x)=sum a_i x_i + sum b_ij x_i x_j to x_i=(1-Z_i)/2."""
    n_qubits = len(linear)
    constant = 0.5 * sum(linear.values()) + 0.25 * sum(quadratic.values())
    fields = {i: -0.5 * linear[i] for i in range(n_qubits)}
    for (i, j), value in quadratic.items():
        fields[i] -= 0.25 * value
        fields[j] -= 0.25 * value

    hamiltonian = QubitOperator((), constant)
    for i, value in fields.items():
        if abs(value) > 1e-12:
            hamiltonian += QubitOperator(f"Z{i}", value)
    for (i, j), value in quadratic.items():
        if abs(value) > 1e-12:
            hamiltonian += QubitOperator(f"Z{i} Z{j}", 0.25 * value)
    return hamiltonian

In [ ]:
n_candidates = len(candidates)
conflict_penalty = 0.60
linear = {i: -scores[i] for i in range(n_candidates)}
quadratic = {pair: conflict_penalty for pair in conflicts}
cost_hamiltonian = qubo_to_ising(linear, quadratic)


def tracking_cost(bits):
    return sum(linear[i] * bits[i] for i in linear) + sum(
        value * bits[i] * bits[j] for (i, j), value in quadratic.items()
    )


ising_diagonal = cost_hamiltonian.sparse_matrix(n_candidates).diagonal().real
for bits in itertools.product([0, 1], repeat=n_candidates):
    basis_index = sum(bit << qubit for qubit, bit in enumerate(bits))
    assert np.isclose(ising_diagonal[basis_index], tracking_cost(bits))

classical_solutions = sorted(
    (tracking_cost(bits), bits) for bits in itertools.product([0, 1], repeat=n_candidates)
)
ground_energy, exact_bits = classical_solutions[0]
print("Exact optimum (q0 first):", exact_bits)
print("Selected candidates:", [candidates[i] for i, bit in enumerate(exact_bits) if bit])
assert exact_bits == (1, 0, 0, 0, 0, 0, 0, 1)

## 3. Run depth-1 QAOA

This small instance already illustrates approximation: the optimized QAOA state is a distribution, not a deterministic optimizer. We compare its probability of the exact solution to the uniform baseline $1/2^8$. At $p=1$ the correct assignment is enhanced above that baseline, although the empty selection can still be the single most likely string; the deeper circuits below progressively concentrate probability on the optimum.

In [ ]:
optimizer = ScipyOptimizer(
    "Nelder-Mead",
    options={"maxiter": 300, "xatol": 1e-5, "fatol": 1e-7},
)
qaoa = QAOA(
    cost_hamiltonian,
    n_layers=1,
    initial_parameters=np.array([0.5, 0.5]),
    optimizer=optimizer,
    save_energy_history=True,
).build()

minimum_expectation, optimal_parameters = qaoa.run()
print(f"Final expectation value: {minimum_expectation:.6f}")
print("Optimal parameters:", optimal_parameters)

final_state = qaoa.get_final_state_block()
sampler = Sampler(ket=final_state, n_shots=qarp.EXACT)
engine = QarpEngine(n_shots=qarp.EXACT)
engine.build([sampler])
probabilities = engine.run()[0]
ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)

most_likely_bits, solution_probability = ranked[0]
uniform_probability = 1.0 / 2**n_candidates
print("Most likely bit string:", most_likely_bits)
print(f"Exact-solution probability: {probabilities[exact_bits]:.3f}")
print(f"Uniform baseline: {uniform_probability:.3f}")
assert probabilities[exact_bits] > 8 * uniform_probability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(qaoa.energy_history)
axes[0].axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
axes[0].set(
    xlabel="objective evaluation", ylabel=r"$\langle H_C\rangle$", title="QAOA optimization"
)
axes[0].legend()

top = ranked[:10]
labels = ["".join(map(str, bits)) for bits, _ in top]
axes[1].bar(labels, [probability for _, probability in top])
axes[1].set(
    xlabel="bit string ($q_0$ first)", ylabel="probability", title="Largest QAOA probabilities"
)
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Extension: compare $p=2,3,4$

The depth-one calculation above supplies a deterministic warm start. We now add layers successively and compare only $p=2,3,4$. For each new depth, the optimized $\beta$ and $\gamma$ sequences from the preceding circuit are linearly interpolated. The separate history plots show convergence of the classical optimizer at each fixed depth; the summary plots show convergence of the variational family as $p$ increases.

In [ ]:
def interpolate_qaoa_parameters(parameters):
    """Lift [beta_0...beta_p-1, gamma_0...gamma_p-1] from p to p + 1."""
    parameters = np.asarray(parameters, dtype=float)
    depth = len(parameters) // 2

    def interpolate(sequence):
        extended = np.empty(depth + 1)
        extended[0] = sequence[0]
        extended[-1] = sequence[-1]
        for index in range(1, depth):
            weight = index / depth
            extended[index] = weight * sequence[index - 1] + (1.0 - weight) * sequence[index]
        return extended

    return np.concatenate([interpolate(parameters[:depth]), interpolate(parameters[depth:])])


def exact_qaoa_distribution(model):
    depth_sampler = Sampler(ket=model.get_final_state_block(), n_shots=qarp.EXACT)
    depth_engine = QarpEngine(n_shots=qarp.EXACT)
    depth_engine.build([depth_sampler])
    return depth_engine.run()[0]


qaoa_by_depth = {}
energy_by_depth = {}
history_by_depth = {}
probability_by_depth = {}
ground_probability_by_depth = {}
layerwise_parameters = np.asarray(optimal_parameters)

for depth in (2, 3, 4):
    layerwise_parameters = interpolate_qaoa_parameters(layerwise_parameters)
    depth_optimizer = ScipyOptimizer(
        "Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-7, "fatol": 1e-10},
    )
    depth_qaoa = QAOA(
        cost_hamiltonian,
        n_layers=depth,
        initial_parameters=layerwise_parameters,
        optimizer=depth_optimizer,
        save_energy_history=True,
    ).build()
    depth_energy, layerwise_parameters = depth_qaoa.run()
    depth_probabilities = exact_qaoa_distribution(depth_qaoa)

    qaoa_by_depth[depth] = depth_qaoa
    energy_by_depth[depth] = float(depth_energy)
    history_by_depth[depth] = np.asarray(depth_qaoa.energy_history)
    probability_by_depth[depth] = depth_probabilities
    ground_probability_by_depth[depth] = depth_probabilities[exact_bits]

depths = np.array(sorted(qaoa_by_depth))
for depth in depths:
    gap = energy_by_depth[depth] - ground_energy
    probability = ground_probability_by_depth[depth]
    print(
        f"p={depth}: energy={energy_by_depth[depth]:.6f}, "
        f"gap={gap:.6f}, exact-track probability={probability:.3f}"
    )

most_likely_bits = max(probability_by_depth[4], key=probability_by_depth[4].get)
assert most_likely_bits == exact_bits
assert energy_by_depth[4] - ground_energy < 0.40
assert ground_probability_by_depth[4] > 0.80

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7), sharey=True)
for axis, depth in zip(axes, depths, strict=True):
    axis.plot(history_by_depth[depth], color=f"C{depth - 2}")
    axis.axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
    axis.set(
        xlabel="objective evaluation",
        title=f"Fixed-depth optimization: p={depth}",
    )
axes[0].set_ylabel(r"$\langle H_C\rangle$")
axes[0].legend()
plt.tight_layout()
plt.show()

final_energies = np.array([energy_by_depth[depth] for depth in depths])
ground_probabilities = np.array([ground_probability_by_depth[depth] for depth in depths])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(depths, final_energies, marker="o", label="optimized QAOA")
axes[0].axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
axes[0].set(
    xlabel="QAOA depth p",
    ylabel=r"optimized $\langle H_C\rangle$",
    xticks=depths,
    title="Tracking-QUBO energy",
)
axes[0].legend()

axes[1].plot(depths, ground_probabilities, marker="o", color="tab:green")
axes[1].axhline(
    uniform_probability,
    color="0.4",
    linestyle=":",
    label="uniform baseline",
)
axes[1].set(
    xlabel="QAOA depth p",
    ylabel="exact-track probability",
    xticks=depths,
    ylim=(0.0, 1.02),
    title="Correct-track sampling probability",
)
axes[1].legend()
plt.tight_layout()
plt.show()

### Most probable bit strings at each depth

Each panel shows the ten most probable assignments in `q0 first` order. The exact two-track reconstruction `(1, 0, 0, 0, 0, 0, 0, 1)` is highlighted in green. The panels share a vertical scale so that both its changing rank and the concentration of probability with increasing depth remain visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for axis, depth in zip(axes, depths, strict=True):
    ranked_depth = sorted(
        probability_by_depth[depth].items(),
        key=lambda item: item[1],
        reverse=True,
    )
    assert ranked_depth[0][0] == exact_bits
    top_depth = ranked_depth[:10]
    labels = ["".join(map(str, bits)) for bits, _ in top_depth]
    values = [probability for _, probability in top_depth]
    colors = ["tab:green" if bits == exact_bits else "tab:blue" for bits, _ in top_depth]
    axis.bar(labels, values, color=colors)
    axis.set(
        xlabel=r"bit string ($q_0$ first)",
        title=(f"p={depth}: correct assignment = {ground_probability_by_depth[depth]:.1%}"),
    )
    axis.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("probability")
fig.suptitle("Ten most probable track selections (correct assignment in green)")
plt.tight_layout()
plt.show()

## 5. Decode the selected tracks and count resources

A measured bit string is $(x_0,…,x_7)$ in `q0 first` order, with bit $x_i$ referring to candidate triplet $T_i$ in the printed `candidates` list. A `1` accepts that candidate as a reconstructed track; a `0` rejects it. Therefore `(1, 0, 0, 0, 0, 0, 0, 1)` selects the genuine candidates `T0 = (A0, A1, A2)` and `T7 = (B0, B1, B2)` and rejects all six mixed combinations. QAOA returns a probability distribution over such selections, and the detector display below decodes its most likely $p=4$ string. The resource comparison makes the accuracy-versus-depth tradeoff explicit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for radius in radii:
    ax.add_patch(plt.Circle((0, 0), radius, fill=False, color="0.8"))
for name, (layer, phi) in hits.items():
    radius = radii[layer]
    x, y = radius * np.cos(phi), radius * np.sin(phi)
    ax.scatter(x, y, color="tab:gray")
    ax.text(1.04 * x, 1.04 * y, name)

for i, bit in enumerate(most_likely_bits):
    if not bit:
        continue
    candidate = candidates[i]
    xy = []
    for name in candidate:
        layer, phi = hits[name]
        radius = radii[layer]
        xy.append((radius * np.cos(phi), radius * np.sin(phi)))
    xy = np.asarray(xy)
    ax.plot(xy[:, 0], xy[:, 1], linewidth=2.5, label=f"selected T{i}")

ax.set(xlabel="x", ylabel="y", title="Most likely QAOA track selection")
ax.set_aspect("equal")
ax.set_xlim(-3.4, 3.4)
ax.set_ylim(-3.4, 3.4)
ax.legend()
plt.show()

for depth in depths:
    logical = estimate(qaoa_by_depth[depth].get_final_state_block())[Stage.LOGICAL]
    print(
        f"p={depth}: {logical.n_qubits} qubits, logical depth {logical.depth}, "
        f"{logical.n_2q} two-qubit gates"
    )

## Takeaways

- The workflow mirrors a realistic hybrid pipeline: classical hit preprocessing and candidate generation, quantum optimization of a discrete compatibility problem, then classical decoding.
- The independent QUBO enumeration verifies both the optimum and the QUBO-to-Ising conversion.
- The useful scaling question is the number and connectivity of candidate tracks. Reducing candidate combinatorics is as important as improving the quantum optimizer.
- Optimizer convergence at fixed $p$ does not guarantee the ground state. Here the layerwise $p=2,3,4$ runs progressively reduce the energy gap and raise the exact-track sampling probability, at increasing circuit cost.
- Even here QAOA is probabilistic. On larger/noisy instances it must be compared honestly with strong classical tracking and QUBO baselines.